# 本題要找出那些user符合指定數量且有一定比例以上的相同reaction (\#Join, \#GroupBy, \#Partition )
原題目連結: https://leetcode.com/problems/find-emotionally-consistent-users/  

Table: `reactions`  
```
+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| user_id      | int     |
| content_id   | int     |
| reaction     | varchar |
+--------------+---------+
```
(user_id, content_id) is the primary key (unique value) for this table.
Each row represents a reaction given by a user to a piece of content.
Write a solution to identify emotionally consistent users based on the following requirements:
- For each user, count the total number of reactions they have given.  
- Only include users who have reacted to at least 5 different content items.  
- A user is considered emotionally consistent if at least 60% of their reactions are of the same type.  

Return the result table ordered by `reaction_ratio` in descending order and then by `user_id` in ascending order.  

Note:    
- reaction_ratio should be rounded to 2 decimal places  

The result format is in the following example.  

範例:  
Ex1:    
Input:  
`reactions` table:  
```
+---------+------------+----------+
| user_id | content_id | reaction |
+---------+------------+----------+
| 1       | 101        | like     |
| 1       | 102        | like     |
| 1       | 103        | like     |
| 1       | 104        | wow      |
| 1       | 105        | like     |
| 2       | 201        | like     |
| 2       | 202        | wow      |
| 2       | 203        | sad      |
| 2       | 204        | like     |
| 2       | 205        | wow      |
| 3       | 301        | love     |
| 3       | 302        | love     |
| 3       | 303        | love     |
| 3       | 304        | love     |
| 3       | 305        | love     |
+---------+------------+----------+
```
Output:  
```
+---------+-------------------+----------------+
| user_id | dominant_reaction | reaction_ratio |
+---------+-------------------+----------------+
| 3       | love              | 1.00           |
| 1       | like              | 0.80           |
+---------+-------------------+----------------+
```
Explanation:  
- User 1:
    - Total reactions = 5
    - like appears 4 times
    - reaction_ratio = 4 / 5 = 0.80
    -Meets the 60% consistency requirement
- User 2:
    - Total reactions = 5
    - Most frequent reaction appears only 2 times
    - reaction_ratio = 2 / 5 = 0.40
    - Does not meet the consistency requirement
- User 3:
    - Total reactions = 5
    - 'love' appears 5 times
    - reaction_ratio = 5 / 5 = 1.00
    - Meets the consistency requirement

The Results table is ordered by reaction_ratio in descending order, then by user_id in ascending order.

* 解題想法:  
首先找出每個使用者的所有content_id，以及每個reaction的出現次數，接著排序找出最多的reaction然後算出比例，最後檢查是否符合60%的比例以及至少有五個以上的content_id，最後排序之後就是要求的答案

```
# Write your MySQL query statement below
select user_id, reaction as dominant_reaction, reaction_ratio from 
(select *, round(r / c, 2) as reaction_ratio from 
(select a.user_id, a.c, b.reaction, b.r, b.o from
(select user_id, count(distinct(content_id)) as c from reactions group by user_id ) a
join 
(select *, row_number() over (partition by user_id order by r desc) as o from 
(select user_id, reaction, count(reaction) as r from reactions group by user_id, reaction ) c ) b
on a.user_id = b.user_id ) t
where o = 1 and c >= 5 ) s
where reaction_ratio >= 0.6 order by reaction_ratio desc, user_id;
```